# Q6. PPO：Advantage 与 Clipping


## Advantage 是什么，为什么比 reward 好

直接用 reward $R$ 作为权重有一个问题：如果所有动作的 reward 都是正数（比如 $+5, +3, +7$），Policy Gradient 会**同时提高所有动作的概率**，训练信号模糊，收敛很慢。

**Advantage 的定义**：

$$A = R - \text{baseline} \qquad (\text{baseline 通常是平均 reward})$$

给定三个动作的 reward $= [+5, +3, +7]$，baseline $= 5$：

1. 计算三个动作的 advantage
2. 解释：advantage 为负的动作，概率应该怎么变？这比直接用 reward 好在哪里？

**验收标准**：能说清楚 advantage 是「比平均水平好多少」，而非绝对好坏。

### 解答

**Step 1：计算 advantage**

| 动作 | reward $R$ | baseline | advantage $A = R - \text{baseline}$ |
|------|-----------|----------|--------------------------------------|
| $a_1$ | +5 | 5 | $0$ |
| $a_2$ | +3 | 5 | $-2$ |
| $a_3$ | +7 | 5 | $+2$ |

**Step 2：advantage 为负时概率怎么变**

Policy Gradient 的更新方向是 $A \cdot \nabla \log \pi$：
- $A > 0$（$a_3$）：梯度为正 → **提高** $a_3$ 的概率（比平均好，多选）
- $A = 0$（$a_1$）：梯度为零 → 概率**不变**（恰好等于平均水平，无信号）
- $A < 0$（$a_2$）：梯度为负 → **降低** $a_2$ 的概率（比平均差，少选）

**Step 3：为什么比直接用 reward 好**

直接用 $R = [+5, +3, +7]$：三个值全为正，模型**同时**被告知提高三个动作的概率，互相竞争，信号混乱，只有靠概率归一化来间接压制差的动作，收敛慢。

用 $A = [0, -2, +2]$：信号变得清晰——**明确告诉模型 $a_2$ 该降、$a_3$ 该升**，训练效率更高。

> Advantage 衡量的是「比平均水平好多少」，而非「绝对好坏」。reward 全为正不代表所有动作都值得鼓励，低于平均的动作同样应该被抑制。